# JupyterHub Mailout Notebook

This notebook fetches resource and user information from a JSON API and sends notifications.

In [ ]:
import datetime
import requests

from mailout_helper import MailoutHelper

In [ ]:
#JUPYTERHUB_API_URL = 'https://binder.rc.nectar.org.au/hub/api'
JUPYTERHUB_API_URL = 'https://jupyterhub.rc.nectar.org.au/hub/api'
JUPYTERHUB_API_TOKEN = ''
SIX_MONTHS_IN_DAYS = 180

## Date/Time Configuration

Set the start and end times for the notification period.

In [ ]:
start_time = '2026-03-01 09:00:00'
end_time = '2026-03-01 17:00:00'
timezone = 'Australia/Melbourne'

## Setup MailoutHelper

In [ ]:
helper = MailoutHelper(
    start_time=start_time,
    end_time=end_time,
    timezone=timezone,
)

## Setup OpenStack

In [ ]:
helper.setup_openstack('https://identity.rctest.nectar.org.au/')

## JupyterHub API access

This helper code implements access to the JupyterHub API and can fetch users

In [ ]:
def _jupyterhub_api(path, *, limit=200, order_by="last_activity", order="desc"):
    """
    Fetch all items from a paginated JupyterHub list endpoint,
    sorted by last_activity (descending by default).
    """
    url = JUPYTERHUB_API_URL.rstrip("/") + path
    headers = {
        "Accept": "application/jupyterhub-pagination+json",
        "Authorization": f"token {JUPYTERHUB_API_TOKEN}",
    }

    params = {
        "limit": limit,
        "offset": 0,
        "order_by": order_by,
        "order": order,
    }

    items = []
    while True:
        r = requests.get(url, headers=headers, params=params)
        r.raise_for_status()
        data = r.json()

        items.extend(data["items"])

        nxt = data["_pagination"]["next"]
        if not nxt:
            break

        params["offset"] = nxt.get("offset", params["offset"] + params["limit"])
        params["limit"] = nxt.get("limit", params["limit"])
        url = nxt.get("url", url)

    return items


def get_recent_users():
    """Return only users with last_activity within the past `months` months."""
    cutoff = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=SIX_MONTHS_IN_DAYS)
    users = _jupyterhub_api("/users")
    recent_users = [
        u for u in users
        if u.get("last_activity") and datetime.datetime.fromisoformat(u['last_activity']) >= cutoff
    ]
    return recent_users

## Fetch the JupyterHub users and extract their email addresses

In [ ]:
hubusers = get_recent_users()
emails = [u['name'] for u in hubusers]

## Define Subject and Body

In [ ]:
# Subject for notfications
subject = 'Test notification for BinderHub'

# Body
body = """{% include 'fragments/greeting.j2' %}
<br>
<br>
This email is to inform you of a scheduled compute host outage to the Nectar
Research Cloud.
{% include 'fragments/schedule.j2' %}
<br>
<br>
<h2>Description</h2>
<br>
A {% if days > 0 %}{{ days }} day {% elif hours > 0 %}{{ hours }} hour {% else %} short {% endif %}
outage is required to perform essential maintenance on the cloud infrastructure
that some of your instances or databases are hosted on.
<br>
<br>
<h2>Impact</h2>
<br>
Affected compute and databases instances will be shut down and will be
inaccessible during the outage.  (Note that they will <b>not</b> be destroyed.)
In addition, management requests (e.g. 'reboot', 'resize' and 'snapshot')
for the affected instances will not work during the outage.
<br>
<br>
<h2>Actions required</h2>
<br>
<br>
Testing only
{% include 'fragments/signoff.j2' %}"""


## Generate and Send Notifications

In [ ]:
notifications = []

context = helper.build_context()
rendered_body = helper.render_template_string(body, context)
rendered_subject = helper.render_template_string(subject, context)

for email in emails:
    notifications.append({
        'subject': rendered_subject,
        'body': rendered_body,
        'to': email,
    })
print(f"Found {len(notifications)} notifications")

## Preview Notifications

In [ ]:
# Preview first notification
helper.preview_notification(notifications[0])

In [ ]:
# Are you ready to send these?
for notification in notifications[3:4]:
    print(f"Sending notification to {notification['to']}...")
    #helper.send_notification(notification)

print(f"Finished")